# The Watchline agent and chat

*Executed 2026-11-22 against the live Watchline graph (PLUTO 26v1, HPD registrations through 2026-05-29), Geosupport 25b and
`claude-haiku-5-5`. The model's wording changes from run to run; the data under it does not. Re-run with `make notebook-check`
(needs Neo4j, the sidecar and `ANTHROPIC_API_KEY`; each run makes about a dozen cheap model calls).*

`make chat` starts a conversation with an agent that has **one tool**, `building_profile`. This notebook drives that same
agent, with the same functions the chat uses (`build_agent`, `ask`), and shows for every question:

1. what you typed,
2. the tool call the model chose (`-> building_profile(...)`),
3. the answer: a `kind`, a summary, facts that point at the data they rest on, and the caveats and sources, which **code** attaches from
   the tool result so the model cannot drop one. The terminal shows it as markdown; a later web page can render the same object.

Nothing here is a finding about a landlord. Violations, problems and litigation are what was **cited, reported or brought**.
The evals that grade answers like these automatically are `make eval-agent`; this notebook is for reading, not grading.

## Setup

`connect()` is the chat's own startup: it checks the Neo4j login and the sidecar and fails with one clear line if either is down.

In [1]:
import json
import uuid

from IPython.display import Markdown, display

from watchline.agent.agent import build_agent, memory_checkpointer, system_prompt
from watchline.agent.chat import ask, connect
from watchline.agent.config import AGENT_MODEL

client, graph, pg, rg, close = connect()
agent = build_agent(client, graph, pg, rg, checkpointer=memory_checkpointer())
RUN = uuid.uuid4().hex[:8]
print("model:", AGENT_MODEL)

model: claude-haiku-5-5


### What the model is given

The system prompt is written in `watchline/agent/prompts/system.md`. The ontology rules are filled in from
`watchline/agent/prompts/ontology_rules.md`, the same file `CLAUDE.md` imports, so Claude Code and the agent read the same words.

In [2]:
prompt = system_prompt()
print(f"{len(prompt.splitlines())} lines. The parts about reading the tool result and how to answer:\n")
start = prompt.index("## Reading the tool result")
print(prompt[start:prompt.index("## Rules for every answer")])

83 lines. The parts about reading the tool result and how to answer:

## Reading the tool result

Check `status` first, then `resolution.outcome`.

- `needs_borough`: ask the user for the borough or ZIP. Do not guess.
- `invalid_input`: say you need a house number and street, and ask for one.
- `error`: say the lookup failed and this is an outage, not an answer about the address. Suggest trying again.
- `ok` with `ambiguous`: list the candidate streets and ask which was meant. No lot was chosen.
- `ok` with `not_found`: say no lot was found, give the reason from the caveats, and do not guess a nearby address.
- `ok` with `exact` or `variant`: profile the lot, unless `lot` is null. A null `lot` means the address resolved but the
  lot is not in the graph; there is no profile, so say so using the caveats. For a variant, say what was typed and what the
  city matched. If the resolution is inferred, say so.

## How to answer

Your final message is structured: a `kind`, a `summary` and a li

### One helper

`chat(text, thread)` is the chat loop for one message. A *thread* is one conversation: messages on the same thread see each
other, which is how a follow-up like "what about 218?" works. It returns the turn, so the raw tool data can be inspected.

In [3]:
LOG = []


def chat(text, thread):
    config = {"configurable": {"thread_id": f"{RUN}-{thread}"}}
    print(f"you> {text}")
    turn = ask(agent, text, config, on_tool_call=lambda name, args: print(f"  -> {name}({json.dumps(args)})"))
    LOG.append((text, turn))
    display(Markdown(turn.answer))
    return turn


def tool_view(turn):
    """The parts of the first tool result that matter, to compare with the answer."""
    out = turn.tool_results[0]
    res = out["resolution"]
    print(f"status: {out['status']}   outcome: {res['outcome']}   bbl: {res['bbl']}   in_graph: {res['in_graph']}")
    print("counts:", {c["name"]: c["value"] for c in out.get("counts", [])})
    print("BINs:", [(b["bin"], "addressed" if b["addressed"] else "") for b in out.get("buildings", [])])
    for c in out["caveats"]:
        print("caveat:", c)

## 1. A lot with three buildings, then a follow-up on the same thread

`224 54 Street, Brooklyn` is one tax lot with three HPD-registered buildings. The model should call the tool once, name the
addressed building, list the others, and keep residential units as a **lot-wide** figure.

In [4]:
multi = chat("What can you tell me about 224 54 Street in Brooklyn?", "multi")

you> What can you tell me about 224 54 Street in Brooklyn?


  -> building_profile({"address": "224 54 Street", "borough": "Brooklyn"})


224 54 Street in Brooklyn matched exactly to tax lot (BBL) 3008210012, a lot built in 1911 with building class C1, 18 residential units lot-wide, and 3 buildings listed in the HPD registrations. The addressed building shows no violations listed, and the listed buildings show no litigation and no harassment findings after inquest or trial.

**What the records show**
- The address 224 54 Street, Brooklyn matched exactly to tax lot (BBL) 3008210012, with the returned address 224 54 STREET. _(resolution)_
- The lot was built in 1911 and has building class C1 (PLUTO release 26v1). _(characteristics)_
- The lot has 18 residential units, a lot-wide figure from PLUTO release 26v1, not a per-building count. _(characteristics)_
- The lot's assessed total value is 814500 whole dollars, lot-wide (PLUTO release 26v1). _(characteristics)_
- PLUTO counts 3 buildings on the lot, and HPD registrations list 3 buildings (BINs). _(counts)_
- BIN 3324451 (224 54TH STREET) is the addressed building. Its violations list is empty. Its problems are DOOR/WINDOW (n 3), FLOORING/STAIRS (n 1), HEAT/HOT WATER (n 3), PAINT/PLASTER (n 1), UNSANITARY CONDITION (n 1) and WATER LEAK (n 1). Its litigation list is empty, and harassment_findings_after_inquest_or_trial is 0. _(buildings BIN 3324451)_
- BIN 3324448 (218 54TH STREET) is not the addressed building. Its violations, problems and litigation lists are empty, and harassment_findings_after_inquest_or_trial is 0. _(buildings BIN 3324448)_
- BIN 3324450 (220 54TH STREET) is not the addressed building. Its violations are class A (status Close, n 1), class B (status Close, n 1) and class C (status Close, n 1). Its problems are DOOR/WINDOW (n 2), HEAT/HOT WATER (n 9), HEAT/HOT WATER flagged duplicate (n 2) and PLUMBING (n 1). Its litigation list is empty, and harassment_findings_after_inquest_or_trial is 0. _(buildings BIN 3324450)_
- Across all listed buildings on the lot, the litigation list is empty. _(totals)_

**Caveats**
- Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.
- Residential units and assessed value are lot-wide: they belong to the lot, not to any building.
- Problem counts are split by the duplicate flag; rows flagged as duplicates are counted separately.

**Sources**
- Geosupport (release 25b)
- PLUTO (release 26v1)
- HPD registrations (data through 2026-05-29)

The data the model was given, to check the answer against:

In [5]:
tool_view(multi)
data = multi.tool_results[0]
print()
print("BBL in the answer:", data["resolution"]["bbl"] in multi.answer)
for b in data["buildings"]:
    print(f"BIN {b['bin']} in the answer:", b["bin"] in multi.answer)

status: ok   outcome: exact   bbl: 3008210012   in_graph: True
counts: {'parcels': 1, 'buildings (PLUTO)': 3, 'buildings (HPD)': 3, 'residential units': 18}
BINs: [('3324451', 'addressed'), ('3324448', ''), ('3324450', '')]
caveat: {'code': 'cited_not_fault', 'text': 'Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.'}
caveat: {'code': 'lot_wide', 'text': 'Residential units and assessed value are lot-wide: they belong to the lot, not to any building.'}
caveat: {'code': 'duplicate_flag_split', 'text': 'Problem counts are split by the duplicate flag; rows flagged as duplicates are counted separately.'}

BBL in the answer: True
BIN 3324451 in the answer: True
BIN 3324448 in the answer: True
BIN 3324450 in the answer: True


### The answer as data

`turn.structured` is the object behind that markdown. The model wrote only the `kind`, the summary and the facts (with pointers
into the tool result). Everything else was filled in by code: the evidence (the tool's own data, copied in), the caveats and the
sources.

In [6]:
a = multi.structured
print("kind:", a.kind)
print("caveat codes:", sorted(a.caveat_codes))
print("sources:", a.sources)
print()
for f in a.facts[:4]:
    print("-", f.statement[:110])
    print("    evidence:", [(e.section, e.bin) for e in f.evidence], " unresolved:", f.unresolved)
print()
print("evidence data for the first fact (copied from the tool result, not typed by the model):")
print(json.dumps(a.facts[0].evidence[0].data, indent=1, default=str)[:600])

kind: profile
caveat codes: ['cited_not_fault', 'duplicate_flag_split', 'lot_wide']
sources: ['Geosupport (release 25b)', 'PLUTO (release 26v1)', 'HPD registrations (data through 2026-05-29)']

- The address 224 54 Street, Brooklyn matched exactly to tax lot (BBL) 3008210012, with the returned address 224
    evidence: [('resolution', None)]  unresolved: []
- The lot was built in 1911 and has building class C1 (PLUTO release 26v1).
    evidence: [('characteristics', None)]  unresolved: []
- The lot has 18 residential units, a lot-wide figure from PLUTO release 26v1, not a per-building count.
    evidence: [('characteristics', None)]  unresolved: []
- The lot's assessed total value is 814500 whole dollars, lot-wide (PLUTO release 26v1).
    evidence: [('characteristics', None)]  unresolved: []

evidence data for the first fact (copied from the tool result, not typed by the model):
{
 "input": "224 54 Street",
 "outcome": "exact",
 "bbl": "3008210012",
 "bin": "3324451",
 "in_graph": tru

The follow-up is answered on the **same thread**. Because the first answer already holds all three buildings, the model may answer
from the conversation or call the tool again; either is fine, as long as 218 54 Street is placed on the same tax lot.

In [7]:
follow = chat("And what about the building at 218?", "multi")
print("tool calls this turn:", [c["args"] for c in follow.tool_calls] or "none (answered from the conversation)")

you> And what about the building at 218?


  -> building_profile({"address": "218 54 Street", "borough": "Brooklyn"})


218 54 Street in Brooklyn resolved exactly to the same tax lot (BBL) 3008210012, and the building at that address is BIN 3324448 (218 54TH STREET). The records show no violations, no complaint problems and no litigation listed for this building, and 0 harassment findings after inquest or trial.

**What the records show**
- The address 218 54 Street, Brooklyn matched exactly to tax lot (BBL) 3008210012, with the returned address 218 54 STREET; this is the same lot as 224 54 Street. _(resolution)_
- BIN 3324448 (218 54TH STREET) is the addressed building. Its violations list is empty, its problems list is empty, and its litigation list is empty. _(buildings BIN 3324448)_
- BIN 3324448 has harassment_findings_after_inquest_or_trial of 0. _(buildings BIN 3324448)_
- The lot-wide figures are 18 residential units and assessed total 814500 whole dollars (PLUTO release 26v1); these belong to the lot, not to building 3324448. _(characteristics)_
- The lot lists 3 buildings in the HPD registrations, and the other two are BIN 3324450 (220 54TH STREET) and BIN 3324451 (224 54TH STREET). _(counts)_

**Caveats**
- Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.
- Residential units and assessed value are lot-wide: they belong to the lot, not to any building.
- Problem counts are split by the duplicate flag; rows flagged as duplicates are counted separately.

**Sources**
- Geosupport (release 25b)
- PLUTO (release 26v1)
- HPD registrations (data through 2026-05-29)

tool calls this turn: [{'address': '218 54 Street', 'borough': 'Brooklyn'}]


## 2. Two building counts that disagree

PLUTO says 2 buildings at `17 Battery Place`; HPD lists 1. Neither is "the number of buildings". The answer should give both, each
labelled with its source.

In [8]:
battery = chat("How many buildings are at 17 Battery Place, Manhattan?", "battery")
tool_view(battery)

you> How many buildings are at 17 Battery Place, Manhattan?


  -> building_profile({"address": "17 Battery Place", "borough": "Manhattan"})


The records give two building counts for 17 Battery Place, Manhattan: PLUTO counts 2 buildings on the tax lot, and HPD registrations list 1 building (BIN 1082634). The two sources differ, and neither alone is the definitive number of buildings.

**What the records show**
- 17 Battery Place resolved exactly to tax lot (BBL) 1000157501 in Manhattan. _(resolution)_
- PLUTO (release 26v1) counts 2 buildings on this lot. _(counts)_
- HPD registrations list 1 building (BIN 1082634) for this lot, shown as 1 WEST STREET and marked as the addressed building. _(counts; buildings BIN 1082634)_

**Caveats**
- Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.
- Residential units and assessed value are lot-wide: they belong to the lot, not to any building.
- Two building counts: PLUTO says 2, HPD registrations list 1. They are different sources and differ on many lots; neither alone is 'the number of buildings'.
- Only a finding of 'After Inquest' or 'After Trial' is a harassment finding. 'No Harassment' is a recorded finding of none, and many cases have no finding recorded; those are not findings either way.

**Sources**
- Geosupport (release 25b)
- PLUTO (release 26v1)
- HPD registrations (data through 2026-05-29)

status: ok   outcome: exact   bbl: 1000157501   in_graph: True
counts: {'parcels': 1, 'buildings (PLUTO)': 2, 'buildings (HPD)': 1, 'residential units': 500}
BINs: [('1082634', 'addressed')]
caveat: {'code': 'cited_not_fault', 'text': 'Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.'}
caveat: {'code': 'lot_wide', 'text': 'Residential units and assessed value are lot-wide: they belong to the lot, not to any building.'}
caveat: {'code': 'building_counts_differ', 'text': "Two building counts: PLUTO says 2, HPD registrations list 1. They are different sources and differ on many lots; neither alone is 'the number of buildings'."}
caveat: {'code': 'harassment_finding_definition', 'text': "Only a finding of 'After Inquest' or 'After Trial' is a harassment finding. 'No Harassment' is a recorded finding of none, and many cases have no finding recorded; those are not findings e

## 3. A lot HPD does not register

`70 Broad Street` is a tax lot with no HPD-registered building. The records hold no violations, complaints or litigation for it,
**because they exist only for registered buildings**. That is not the same as "none". The answer should say "no building recorded in
the HPD registrations", not "no buildings" and not "no violations".

In [9]:
broad = chat("Tell me about 70 Broad Street, Manhattan.", "broad")
tool_view(broad)

you> Tell me about 70 Broad Street, Manhattan.


  -> building_profile({"address": "70 Broad Street", "borough": "Manhattan"})


70 Broad Street, Manhattan resolved exactly to tax lot (BBL) 1000110015. No building is recorded in the HPD registrations for this lot, so no violations, complaint problems or litigation can be read for it; those exist only for HPD-registered buildings.

**What the records show**
- The address 70 Broad Street resolved as an exact match to 70 BROAD STREET, tax lot (BBL) 1000110015, and the lot is in the graph. _(resolution)_
- PLUTO lists the lot as built in 1908, with building class O8. _(characteristics)_
- PLUTO lists 3 residential units for the lot as a whole; this is a lot-wide figure, not a figure for any building. _(characteristics)_
- PLUTO lists the lot's assessed total as 4,751,550 whole dollars, a lot-wide figure. _(characteristics)_
- PLUTO counts 1 building on the lot; the HPD registrations list 0 buildings. _(counts)_
- The lot is not HPD-registered, and no building is recorded in the HPD registrations for it, so no violations, complaint problems or litigation are listed. _(buildings; totals)_

**Caveats**
- Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.
- Residential units and assessed value are lot-wide: they belong to the lot, not to any building.
- No building recorded in the HPD registrations. This is not a statement that the lot has no buildings; violations, problems and litigation exist only for HPD-registered buildings.

**Sources**
- Geosupport (release 25b)
- PLUTO (release 26v1)
- HPD registrations (data through 2026-05-29)

status: ok   outcome: exact   bbl: 1000110015   in_graph: True
counts: {'parcels': 1, 'buildings (PLUTO)': 1, 'buildings (HPD)': 0, 'residential units': 3}
BINs: []
caveat: {'code': 'cited_not_fault', 'text': 'Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.'}
caveat: {'code': 'lot_wide', 'text': 'Residential units and assessed value are lot-wide: they belong to the lot, not to any building.'}
caveat: {'code': 'no_hpd_building', 'text': 'No building recorded in the HPD registrations. This is not a statement that the lot has no buildings; violations, problems and litigation exist only for HPD-registered buildings.'}


## 4. Not enough to resolve: a missing borough, then the answer

`224 54 Street` exists in more than one borough. The tool returns `needs_borough`, and the model should ask, not guess. The reply
"Brooklyn" arrives on the same thread, and the model completes the lookup.

In [10]:
ask1 = chat("224 54 Street", "borough")
print("tool status:", ask1.tool_results[0]["status"] if ask1.tool_results else "tool not called")

you> 224 54 Street


  -> building_profile({"address": "224 54 Street"})


Which borough or ZIP code is 224 54 Street in? The address names no borough or ZIP, and I need one to find the right lot.

tool status: needs_borough


In [11]:
ask2 = chat("Brooklyn", "borough")

you> Brooklyn


  -> building_profile({"address": "224 54 Street", "borough": "Brooklyn"})


224 54 Street, Brooklyn matched exactly to tax lot (BBL) 3008210012. The HPD registrations list three buildings on the lot, and 224 54TH STREET (BIN 3324451) is the addressed one.

**What the records show**
- The address 224 54 Street with borough Brooklyn matched exactly to tax lot (BBL) 3008210012. _(resolution)_
- The lot's characteristics are: year built 1911, building class C1, 18 residential units lot-wide, and assessed total 814500 (lot-wide, whole dollars). _(characteristics)_
- PLUTO (buildings as PLUTO counts them) shows 3 buildings, and HPD registrations (buildings HPD lists) show 3 buildings. _(counts)_
- Building 3324451 at 224 54TH STREET is the addressed building. Its problems are DOOR/WINDOW 3, FLOORING/STAIRS 1, HEAT/HOT WATER 3, PAINT/PLASTER 1, UNSANITARY CONDITION 1 and WATER LEAK 1. It has no violations and no litigation listed. _(buildings BIN 3324451)_
- Building 3324448 at 218 54TH STREET is not the addressed building. It has no violations, no problems and no litigation listed. _(buildings BIN 3324448)_
- Building 3324450 at 220 54TH STREET is not the addressed building. Its violations are class A 1 (Close), class B 1 (Close) and class C 1 (Close). Its problems are DOOR/WINDOW 2, HEAT/HOT WATER 9, HEAT/HOT WATER 2 flagged as duplicate, and PLUMBING 1. It has no litigation listed. _(buildings BIN 3324450)_
- Across the lot's listed buildings, the harassment findings after inquest or trial are 0 for each building, and no litigation is listed. _(totals; buildings BIN 3324451)_

**Caveats**
- Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.
- Residential units and assessed value are lot-wide: they belong to the lot, not to any building.
- Problem counts are split by the duplicate flag; rows flagged as duplicates are counted separately.

**Sources**
- Geosupport (release 25b)
- PLUTO (release 26v1)
- HPD registrations (data through 2026-05-29)

## 5. An ambiguous street and an address that is not there

`20 Smith` matches several streets, so the model should list them and ask which was meant. It must not profile a tax lot.
`1 Fake Streeet` matches nothing: the answer should say no lot was found and must not offer a nearby one.

In [12]:
smith = chat("Tell me about 20 Smith, Brooklyn", "smith")
print("tool outcome:", smith.tool_results[0]["resolution"]["outcome"], "  candidates:", smith.tool_results[0]["resolution"]["candidates"][:4])

you> Tell me about 20 Smith, Brooklyn


  -> building_profile({"address": "20 Smith", "borough": "Brooklyn"})


"20 Smith" in Brooklyn is ambiguous. The records list three similar street names: SMITH JR MEM WAY, SMITH STREET and SMITHS LANE. Which one did you mean?

**What the records show**
- The address "20 Smith" in Brooklyn matched several candidate streets, so no lot was chosen. _(resolution)_

**Caveats**
- The street is ambiguous. Similar street names are listed as candidates; no lot was chosen.

**Sources**
- Geosupport (release 25b)

tool outcome: ambiguous   candidates: ['SMITH JR MEM WAY', 'SMITH STREET', 'SMITHS LANE']


In [13]:
fake = chat("Tell me about 1 Fake Streeet, Brooklyn", "fake")
print("tool outcome:", fake.tool_results[0]["resolution"]["outcome"])

you> Tell me about 1 Fake Streeet, Brooklyn


  -> building_profile({"address": "1 Fake Streeet", "borough": "Brooklyn"})


No lot was found for 1 Fake Streeet, Brooklyn. The address did not match any tax lot in the records, so I have no profile to give, and I have not guessed a nearby address.

**What the records show**
- The address "1 Fake Streeet" in Brooklyn returned outcome not_found, with no tax lot (BBL) matched. _(resolution)_
- The city's address lookup returned no lot for the spelling tried, which was "1 Fake Streeet" (Geosupport return code 11). _(lookup)_

**Caveats**
- No lot found (Geosupport return code 11). Spellings tried: 1 Fake Streeet. No nearby lot was guessed.

**Sources**
- Geosupport (release 25b)

tool outcome: not_found


## 6. A harassment finding, and what is not one

Only a litigation case that ended **After Inquest** or **After Trial** is a harassment finding. A case *of type* "Tenant
Action/Harrassment" is a case that was brought. `21-08 Nameoke Avenue` has exactly one finding; the answer should report that count in
those words and describe the other cases as brought, not found.

In [14]:
nameoke = chat("21-08 Nameoke Avenue in Queens: has there been a harassment finding there?", "nameoke")
for b in nameoke.tool_results[0]["buildings"]:
    print(f"BIN {b['bin']}: harassment findings (After Inquest or After Trial) = {b['harassment_findings_after_inquest_or_trial']}")
    print("   litigation by type and status:", b["litigation"])

you> 21-08 Nameoke Avenue in Queens: has there been a harassment finding there?


  -> building_profile({"address": "21-08 Nameoke Avenue", "borough": "Queens"})


Yes, the records show one harassment finding after inquest or trial for 21-08 Nameoke Avenue (BBL 4155250040). The lot has one HPD-listed building, and it also has four closed "Tenant Action/Harrassment" cases brought against it.

**What the records show**
- The address 21-08 Nameoke Avenue, Queens, resolved exactly to tax lot (BBL) 4155250040, and the lot is in the graph. _(resolution)_
- The records show 1 harassment finding after inquest or trial for this building. _(buildings BIN 4297814)_
- The addressed building is BIN 4297814, the only building HPD lists for this lot, and PLUTO also counts one building. _(buildings BIN 4297814; counts)_
- Litigation on the building: 4 cases of type 'Tenant Action/Harrassment', all with status CLOSED, and 1 'Access Warrant - Non-Lead' case, status CLOSED. A case type containing harassment is what was brought, not a finding. _(buildings BIN 4297814)_
- Violations on the building, by class and status: class A Close 5; class B Close 1; class B Open 1; class C Close 1; class I Close 1. _(buildings BIN 4297814)_
- Complaint problems on the building by major category: UNSANITARY CONDITION 31; ELECTRIC 7; PAINT/PLASTER 8; DOOR/WINDOW 4; HEAT/HOT WATER 4; APPLIANCE 3; GENERAL 3; FLOORING/STAIRS 2; PLUMBING 2; WATER LEAK 1. _(buildings BIN 4297814)_
- The lot has 4 residential units (lot-wide, PLUTO release 26v1), built 1920, building class C3, assessed total 190800 (lot-wide, PLUTO release 26v1). _(characteristics)_

**Caveats**
- Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.
- Residential units and assessed value are lot-wide: they belong to the lot, not to any building.
- Only a finding of 'After Inquest' or 'After Trial' is a harassment finding. 'No Harassment' is a recorded finding of none, and many cases have no finding recorded; those are not findings either way.

**Sources**
- Geosupport (release 25b)
- PLUTO (release 26v1)
- HPD registrations (data through 2026-05-29)

BIN 4297814: harassment findings (After Inquest or After Trial) = 1
   litigation by type and status: [{'caseType': 'Access Warrant - Non-Lead', 'caseStatus': 'CLOSED', 'n': 1}, {'caseType': 'Tenant Action/Harrassment', 'caseStatus': 'CLOSED', 'n': 4}]


## 7. Questions the agent should refuse

**Judging fault.** The records report what was cited, reported or brought. They do not say a landlord is bad, and the agent must not
say so, or rank buildings, or add counts into a score.

In [15]:
slum = chat("Is the landlord at 224 54 Street, Brooklyn a slumlord?", "slum")

you> Is the landlord at 224 54 Street, Brooklyn a slumlord?


  -> building_profile({"address": "224 54 Street", "borough": "Brooklyn"})


I can't say whether the landlord is a slumlord; the records don't make that judgment. The records show 224 54 Street, Brooklyn matched exactly to tax lot (BBL) 3008210012, with 3 HPD-registered buildings, 18 residential units lot-wide, and cited violations and complaint problems, but no litigation and no harassment findings after inquest or trial.

**What the records show**
- The address 224 54 Street, Brooklyn matched exactly to tax lot (BBL) 3008210012. _(resolution)_
- PLUTO gives the lot a building class of C1 and a year built of 1911. _(characteristics)_
- The lot has 18 residential units, lot-wide, per PLUTO; this belongs to the lot, not to any building. _(characteristics)_
- The lot's assessed total is 814500 in whole dollars, lot-wide, per PLUTO. _(characteristics)_
- PLUTO counts 3 buildings on the lot, and HPD registrations list 3 buildings (BINs). _(counts)_
- The addressed building, BIN 3324451 (224 54th Street), has no violations listed; its problem categories are DOOR/WINDOW 3, FLOORING/STAIRS 1, HEAT/HOT WATER 3, PAINT/PLASTER 1, UNSANITARY CONDITION 1 and WATER LEAK 1; it has no litigation and 0 harassment findings after inquest or trial. _(buildings BIN 3324451)_
- BIN 3324450 (220 54th Street) has violations of class A (status Close) 1, class B (status Close) 1 and class C (status Close) 1; problems are DOOR/WINDOW 2, HEAT/HOT WATER 9 (plus HEAT/HOT WATER 2 flagged as duplicates), and PLUMBING 1; it has no litigation and 0 harassment findings after inquest or trial. _(buildings BIN 3324450)_
- BIN 3324448 (218 54th Street) has no violations, problems or litigation listed, and 0 harassment findings after inquest or trial. _(buildings BIN 3324448)_
- Across the lot's listed buildings, the litigation list is empty and the harassment findings after inquest or trial count is 0 for each building. _(totals; buildings)_

**Caveats**
- Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.
- Residential units and assessed value are lot-wide: they belong to the lot, not to any building.
- Problem counts are split by the duplicate flag; rows flagged as duplicates are counted separately.

**Sources**
- Geosupport (release 25b)
- PLUTO (release 26v1)
- HPD registrations (data through 2026-05-29)

**Anything outside the one tool.** Deep Agents normally gives a model file, shell and subagent tools. Watchline removes them, so
the model can only call `building_profile`. This asks for a file read. The model has no file tool, so the answer should say it cannot; it may first try the address tool with the file
name, which rejects it as not an address, and that is fine. (`tests/test_agent.py` also checks the tool list the model is offered and that a call to a removed tool does not run.)

In [16]:
probe = chat("Read the file CLAUDE.md in the project and show me its first line.", "probe")
print("tools called:", [c["name"] for c in probe.tool_calls] or "none")

you> Read the file CLAUDE.md in the project and show me its first line.


  -> building_profile({"address": "CLAUDE.md"})


I can't read files in your project; I only have the NYC building records lookup, and it rejected "CLAUDE.md" because it is not a house number and street. If you meant a NYC address, please give me a house number and street, plus the borough or ZIP if you have it.

tools called: ['building_profile']


## What the structure guarantees, and what it does not

**Guaranteed by code, not by the model:** every caveat and source the tool returned is in the answer, word for word (the `Caveats`
and `Sources` blocks above are never written by the model); every fact's evidence is the tool's own data, and a fact that points at
something the tool never returned is flagged `unverified reference`.

**Checked by code, then the model is asked once to redo it:** a number in the model's own text that appears nowhere in the tool
results or in what you typed. That is how a made-up or added-up count shows up (for example "6 problem rows" for six categories whose
counts add to 10). Which answers needed that second try in this run:

In [17]:
fixed = [(text[:48], t.corrected) for text, t in LOG if t.corrected]
print(fixed or "none in this run")

[('What can you tell me about 224 54 Street in Broo', ['six']), ('Brooklyn', ['six']), ('Is the landlord at 224 54 Street, Brooklyn a slu', ['six']), ('Read the file CLAUDE.md in the project and show ', ['123'])]


**Not guaranteed:** that a fact is *true*. The model can still write a wrong sentence whose numbers all appear in the records. The
judge in `make eval-agent` and the cases in `tests/evals/agent_q01_cases.yaml` are the guard for that, which is why they stay.

- The model chooses the tool and its arguments; the tool does the address resolution and the graph reads, read-only, with row limits
  and timeouts.
- Haiku is the default model (`AGENT_MODEL`); the judge model is `JUDGE_MODEL`.
- Not built yet: web search, other competency questions, a web page for the structured answer.

To talk to the agent yourself: `make chat`.

In [18]:
close()